# Week 4 Coding Assignment: Data Acquisition — SQL, APIs & Web Data

**Data Science: Foundations to Applications (SCD-4XX) | Kyungdong University, Global Campus**

**Outcome being assessed:** Retrieve data from databases and external sources programmatically.

## Instructions
- Complete every function below by replacing `# TODO` with your code.
- **Always use `?` placeholders in SQL queries that include a variable — never an f-string or `%` formatting.** This is graded as part of the assignment.
- Run the test cell below each function to check your answer.
- When finished: save the notebook, then `git add`, `git commit`, and `git push` to your `data-science-labs-2026` repository, per the Week 1 lab.

**Parts:**
- Part A — SQL Fundamentals (Problems 1–4)
- Part B — Python + SQL Safely (Problems 5–6)
- Part C — REST APIs (Problems 7–9)
- Stretch Challenge (optional) — Ethical Web Scraping


> ⚠️ **Run this notebook in Jupyter Notebook (via Anaconda) — not Google Colab.**
This course uses your local Jupyter environment for all labs and assignments.
Open it the same way you did in Week 1: navigate to your cloned repo folder in a terminal and run `jupyter notebook`.

## Setup — run this first
This builds a sample `retail.db` SQLite database with two linked tables, `customers` and `orders`, that every problem below uses. You don't need to modify this cell.

In [1]:
import sqlite3
import random
import time
import requests
import pandas as pd
from datetime import date, timedelta

random.seed(7)

conn = sqlite3.connect("retail.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders")
cur.execute("DROP TABLE IF EXISTS customers")

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    segment TEXT
)
""")
cur.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cities = ["Seoul", "Busan", "Incheon", "Daegu", "Gwangju"]
segments = ["Retail", "Wholesale", "Online"]
names = ["Kim Minji","Park Jisoo","Lee Jaehyun","Choi Seoyeon","Jung Hyunwoo",
         "Kang Yuna","Yoon Doyoon","Han Soomin","Oh Jiwoo","Shin Taeyang",
         "Baek Narae","Song Junho","Moon Hana","Ahn Jiho","Jang Yerin"]

customers = [(i, name, random.choice(cities), random.choice(segments))
             for i, name in enumerate(names, start=1)]
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

start_date = date(2026, 1, 1)
order_id, orders = 1, []
for c in customers:
    for _ in range(random.randint(2, 8)):
        d = start_date + timedelta(days=random.randint(0, 270))
        amt = round(random.uniform(15000, 450000), -2)
        orders.append((order_id, c[0], d.isoformat(), amt))
        order_id += 1
cur.executemany("INSERT INTO orders VALUES (?,?,?,?)", orders)
conn.commit()

print(f"retail.db ready: {len(customers)} customers, {len(orders)} orders.")

retail.db ready: 15 customers, 73 orders.


---
## Part A — SQL Fundamentals

### Problem 1 — Find Customers by City and Segment
Return the `name`, `city`, and `segment` of every customer in a given `city` and `segment`.

**Hint:** `WHERE ... AND ...`

In [8]:

def customers_in_city_segment(conn, city, segment):
    """
    conn: sqlite3 connection
    city: str, e.g. "Seoul"
    segment: str, e.g. "Online"
    returns: pandas DataFrame with columns name, city, segment
    """

    sql = """
    SELECT name, city, segment
    FROM customers
    WHERE city = ? AND segment = ?
    """

    return pd.read_sql(sql, conn, params=(city, segment))


# Test
print(customers_in_city_segment(conn, "Seoul", "Online"))


         name   city segment
0   Kang Yuna  Seoul  Online
1  Song Junho  Seoul  Online


### Problem 2 — Join Customers with Their Orders
Return `name`, `order_date`, and `amount` for every order, joined with the customer who placed it. Limit to the first 10 rows.

**Hint:** `INNER JOIN ... ON ...`, `LIMIT`

In [9]:
def orders_with_customer_names(conn):
    """
    returns: pandas DataFrame with columns name, order_date, amount (first 10 rows)
    """
    sql = """
    SELECT customers.name, orders.order_date, orders.amount
    FROM customers
    INNER JOIN orders
    ON customers.customer_id = orders.customer_id
    LIMIT 10
    """

    return pd.read_sql(sql, conn)


print(orders_with_customer_names(conn))

         name  order_date    amount
0   Kim Minji  2026-01-26  439700.0
1   Kim Minji  2026-01-24  257100.0
2   Kim Minji  2026-03-10  141000.0
3   Kim Minji  2026-03-15  250200.0
4   Kim Minji  2026-06-07  258700.0
5  Park Jisoo  2026-04-03   59800.0
6  Park Jisoo  2026-04-07  177000.0
7  Park Jisoo  2026-02-02  260500.0
8  Park Jisoo  2026-04-16  230900.0
9  Park Jisoo  2026-08-07  353100.0


### Problem 3 — Total Revenue per Customer (High Spenders Only)
Return each customer's `name` and their `total_spent` (sum of `amount`), but only for customers whose total exceeds a given `min_total`. Sort by `total_spent` descending.

**Hint:** `GROUP BY`, `HAVING`, `ORDER BY ... DESC`

In [11]:
def high_spending_customers(conn, min_total=300000):
    """
    returns: pandas DataFrame with columns name, total_spent
    """
    sql = """
    SELECT c.name, SUM(o.amount) AS total_spent
    FROM customers AS c
    INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.name
    HAVING SUM(o.amount) > ?
    ORDER BY total_spent DESC
    """

    return pd.read_sql(sql, conn, params=(min_total,))
print(high_spending_customers(conn))

            name  total_spent
0     Jang Yerin    2167100.0
1    Yoon Doyoon    1609600.0
2     Baek Narae    1559500.0
3     Park Jisoo    1523300.0
4       Oh Jiwoo    1437300.0
5     Song Junho    1359000.0
6      Kim Minji    1346700.0
7       Ahn Jiho    1129700.0
8      Moon Hana     895700.0
9   Jung Hyunwoo     861200.0
10  Shin Taeyang     619200.0
11   Lee Jaehyun     613200.0
12    Han Soomin     427900.0
13  Choi Seoyeon     374600.0


### Problem 4 — Top N Customers by Spend
Return the top `n` customers by total spend — same idea as Problem 3, but capped to a specific count regardless of threshold.

**Hint:** `LIMIT`

In [14]:
def top_n_customers(conn, n=3):
    """
    returns: pandas DataFrame with columns name, total_spent — exactly n rows"""
    sql = """
    SELECT c.name, SUM(o.amount) AS total_spent
    FROM customers AS c
    INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY total_spent DESC
    LIMIT ?
    """

    return pd.read_sql(sql, conn, params=(n,))


print(top_n_customers(conn, 3))

          name  total_spent
0   Jang Yerin    2167100.0
1  Yoon Doyoon    1609600.0
2   Baek Narae    1559500.0


---
## Part B — Python + SQL Safely

### Problem 5 — Search Orders by Customer Name (Safely!)
Write a function that looks up every order placed by a given customer name. **This must use a `?` placeholder** — it will be checking user-supplied input, exactly the SQL-injection scenario from the demo.

**Hint:** `cursor.execute(sql, (value,))`, then `.fetchall()`

In [15]:
def find_orders_by_customer(conn, customer_name):
    """
    conn: sqlite3 connection
    customer_name: str
    returns: list of tuples (order_id, order_date, amount)
    """
    # TODO: implement using cursor.execute with a ? placeholder — no f-strings!
    sql = """
    SELECT o.order_id, o.order_date, o.amount
    FROM orders AS o
    INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
    WHERE c.name = ?
    """

    cursor = conn.cursor()
    cursor.execute(sql, (customer_name,))
    return cursor.fetchall()


print(find_orders_by_customer(conn, "Kim Minji"))

[(1, '2026-01-26', 439700.0), (2, '2026-01-24', 257100.0), (3, '2026-03-10', 141000.0), (4, '2026-03-15', 250200.0), (5, '2026-06-07', 258700.0)]


### Problem 6 — Monthly Revenue Report
Using `pd.read_sql()`, return total revenue grouped by month (format `YYYY-MM`), ordered chronologically.

**Hint:** SQLite's `strftime('%Y-%m', order_date)` extracts the year-month from a date string.

In [16]:
def monthly_revenue(conn):
    """
    returns: pandas DataFrame with columns month, revenue
    """
    # TODO: implement using pd.read_sql
    sql = """
    SELECT
        strftime('%Y-%m', order_date) AS month,
        SUM(amount) AS revenue
    FROM orders
    GROUP BY strftime('%Y-%m', order_date)
    ORDER BY month ASC
    """

    return pd.read_sql(sql, conn)

print(monthly_revenue(conn))

     month    revenue
0  2026-01  1605200.0
1  2026-02  1924900.0
2  2026-03  1836600.0
3  2026-04  2277600.0
4  2026-05  1005400.0
5  2026-06  2817100.0
6  2026-07  1715100.0
7  2026-08  1906900.0
8  2026-09  1097800.0


---
## Part C — REST APIs

### Problem 7 — Fetch Weather for a City
Write a function that calls the Open-Meteo API (same one from the demo) for a given latitude/longitude and returns the parsed JSON `current_weather` dictionary. Check `response.status_code` and return `None` if the request fails.

**Hint:** `requests.get(url, params=...)`, `response.json()`

In [17]:
def get_current_weather(latitude, longitude):
    """
    returns: dict (the current_weather portion of the response), or None on failure
    """
    url = "https://api.open-meteo.com/v1/forecast"
    # TODO: build params, send the GET request, check status_code, return data["current_weather"]
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": "true"
    }

    try:
        response = requests.get(url, params=params, timeout=15)

        if response.status_code == 200:
            data = response.json()
            return data["current_weather"]
        else:
            return None

    except requests.RequestException:
        return None

print(get_current_weather(37.5665, 126.9780))  # Seoul

{'time': '2026-10-11T01:30', 'interval': 900, 'temperature': 20.3, 'windspeed': 2.9, 'winddirection': 83, 'is_day': 1, 'weathercode': 0}


### Problem 8 — Weather for Multiple Cities into a DataFrame
Using the function from Problem 7, fetch the current weather for a list of `(city_name, lat, lon)` tuples and combine the results into a single DataFrame with a `city` column.

**Hint:** Build a list of dicts (adding a `"city"` key to each result), then `pd.DataFrame(list_of_dicts)`.

In [18]:
def weather_for_cities(cities):
    """
    cities: list of (city_name, latitude, longitude) tuples
    returns: pandas DataFrame with a 'city' column plus the weather fields
    """
    # TODO: implement — loop over cities, call get_current_weather, collect into a DataFrame
    results = []

    for city, lat, lon in cities:
        weather = get_current_weather(lat, lon)

        if weather is not None:
            weather["city"] = city
            results.append(weather)

    return pd.DataFrame(results)


test_cities = [
    ("Seoul", 37.5665, 126.9780),
    ("Busan", 35.1796, 129.0756),
    ("Tokyo", 35.6762, 139.6503)
]

test_cities = [("Seoul", 37.5665, 126.9780), ("Busan", 35.1796, 129.0756), ("Tokyo", 35.6762, 139.6503)]
print(weather_for_cities(test_cities))

               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-11T01:30       900         20.3        2.9             83       1   
1  2026-10-11T01:30       900         24.6        8.0             80       1   
2  2026-10-11T01:30       900         22.9        5.2             12       1   

   weathercode   city  
0            0  Seoul  
1            1  Busan  
2            0  Tokyo  


### Problem 9 — Respect the Rate Limit
Modify your approach from Problem 8 so that your function pauses `delay_seconds` between each API call, exactly as discussed in the demo's "Authentication and Rate Limits" section.

**Hint:** `time.sleep(delay_seconds)` inside your loop, after each request.

In [20]:
def weather_for_cities_paced(cities, delay_seconds=1):
    """
    Same as Problem 8, but pauses delay_seconds between each request.
    returns: pandas DataFrame with a 'city' column plus the weather fields
    """
    # TODO: implement
    results = []

    for i, (city, lat, lon) in enumerate(cities):
        weather = get_current_weather(lat, lon)

        if weather is not None:
            weather["city"] = city
            results.append(weather)

        if i < len(cities) - 1:
            time.sleep(delay_seconds)

    return pd.DataFrame(results)



print(weather_for_cities_paced(test_cities, delay_seconds=1))

               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-11T01:30       900         20.3        2.9             83       1   
1  2026-10-11T01:30       900         24.6        8.0             80       1   
2  2026-10-11T01:30       900         22.9        5.2             12       1   

   weathercode   city  
0            0  Seoul  
1            1  Busan  
2            0  Tokyo  


---
## Stretch Challenge (Optional) — Ethical Web Scraping

Scrape the first page of [books.toscrape.com](http://books.toscrape.com) and return a DataFrame with each book's title and price. Follow the demo's ethics checklist: use a descriptive `User-Agent`, and call `time.sleep(1)` after your request.

**Hint:** `BeautifulSoup`, `soup.select("article.product_pod")`, then pull the title from `h3 a` (the `title` attribute) and the price from `p.price_color` within each matched element.


In [21]:
from bs4 import BeautifulSoup

def scrape_book_titles_and_prices():
    """
    returns: pandas DataFrame with columns title, price
    """
    url = "http://books.toscrape.com"
    headers = {"User-Agent": "KDU-DataScience-Student"}
    # TODO: implement — request, parse with BeautifulSoup, extract title + price, time.sleep(1)
    response = requests.get(url, headers=headers, timeout=15)

    time.sleep(1)

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    books = []

    for item in soup.select("article.product_pod"):
        title = item.select_one("h3 a")["title"]
        price = item.select_one("p.price_color").get_text(strip=True)

        books.append({
            "title": title,
            "price": price
        })

    return pd.DataFrame(books, columns=["title", "price"])



print(scrape_book_titles_and_prices())

                                                title    price
0                                A Light in the Attic  Â£51.77
1                                  Tipping the Velvet  Â£53.74
2                                          Soumission  Â£50.10
3                                       Sharp Objects  Â£47.82
4               Sapiens: A Brief History of Humankind  Â£54.23
5                                     The Requiem Red  Â£22.65
6   The Dirty Little Secrets of Getting Your Dream...  Â£33.34
7   The Coming Woman: A Novel Based on the Life of...  Â£17.93
8   The Boys in the Boat: Nine Americans and Their...  Â£22.60
9                                     The Black Maria  Â£52.15
10     Starving Hearts (Triangular Trade Trilogy, #1)  Â£13.99
11                              Shakespeare's Sonnets  Â£20.66
12                                        Set Me Free  Â£17.46
13  Scott Pilgrim's Precious Little Life (Scott Pi...  Â£52.29
14                          Rip it Up and Start Again  

---
## Cleanup
Run this last to close your database connection.

In [22]:
conn.close()
print("Connection closed.")

Connection closed.
